In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set pandas display options for better visibility in Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [36]:
# Block 2: Define the Audit Function
def audit_table(file_path):
    print(f"{'='*60}")
    print(f" Auditing Table: {file_path}")
    print(f"{'='*60}")
    
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f" File '{file_path}' not found. Please check the file path.")
        return None
        
    # 1. Table Shape
    print(f" 1. Shape: {df.shape[0]:,} rows | {df.shape[1]} columns")
    
    # 2. Check for trailing/leading spaces in column names
    messy_cols = [col for col in df.columns if col != col.strip()]
    if messy_cols:
        print(f"2. Columns with trailing/leading spaces: {messy_cols} ")
    else:
        print("2. Column names are clean.")
        
    # 3. Check for fully duplicated rows
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"3. Fully duplicated rows: {duplicates:,} rows ")
    else:
        print("3. No duplicated rows.")
        
    # 4. Check for missing values (Nulls)
    missing = df.isnull().sum()
    missing_df = missing[missing > 0]
    if not missing_df.empty:
        print(f"4. Missing Values (Nulls):")
        for col, count in missing_df.items():
            print(f"   - {col}: {count:,} nulls ({(count/len(df))*100:.2f}%)")
    else:
        print("4. No missing values.")
        
    # 5. Numeric Summary (Identify impossible values / outliers)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if not num_cols.empty:
        print(f"\n🔹 5. Numeric Summary (Min - Mean - Max):")
        stats = df[num_cols].describe().T[['min', 'mean', 'max']].round(2)
        print(stats.to_string())
        
    # 6. Categorical Summary (Identify typos and inconsistent text)
    cat_cols = df.select_dtypes(include=['object']).columns
    if not cat_cols.empty:
        print(f"\n 6. Categorical Summary:")
        for col in cat_cols:
            unique_count = df[col].nunique()
            print(f"   - {col}: {unique_count} unique values")
            # Print values if categories are 15 or less to spot typos
            if unique_count <= 15:
                print(f"     Values are: {df[col].dropna().unique().tolist()}")
    print("\n")
    
    # Return the dataframe to keep it in memory for the next steps
    return df

In [42]:
# Block 3: Audit Dim_Employees
df_emp = audit_table("D:\\HR Project\\HR data\\Raw\\Dim_Employees.csv")

 Auditing Table: D:\HR Project\HR data\Raw\Dim_Employees.csv
 1. Shape: 15,000 rows | 10 columns
2. Columns with trailing/leading spaces: ['employee_id '] 
3. No duplicated rows.
4. Missing Values (Nulls):
   - exit_date: 12,556 nulls (83.71%)

🔹 5. Numeric Summary (Min - Mean - Max):
              min    mean      max
employee_id  1.00 7500.50 15000.00
manager_id   2.00 7623.41 15099.00
role_id      1.00   37.60    77.00

 6. Categorical Summary:
   - first_name: 329 unique values
   - last_name: 79 unique values
   - gender: 2 unique values
     Values are: ['Male', 'Female']
   - birth_date: 7369 unique values
   - hire_date: 2008 unique values
   - exit_date: 1250 unique values
   - attrition_flag: 2 unique values
     Values are: ['No', 'Yes']




In [38]:
df_emp.head()

,employee_id,first_name,last_name,gender,birth_date,hire_date,exit_date,attrition_flag,manager_id,role_id
0,1,Karim,Ghoneim,Male,07/29/1998,10/17/2022,NaN,No,8785,57
1,2,rabab,Zaki,Female,06/14/1977,04/21/2020,NaN,No,10228,63
2,3,Samy,Shalaby,Male,08/10/1996,07/23/2020,06/10/2025,Yes,12654,47
3,4,Tarek,Nasr,Male,09/28/1976,03/16/2020,NaN,No,7039,74
4,5,Heba,Khamis,Female,10/08/1996,06/25/2020,01/08/2024,Yes,4626,69


In [41]:
df_roles = audit_table("D:\\HR Project\\HR data\\Raw\\Dim_Job_Roles.csv")

 Auditing Table: D:\HR Project\HR data\Raw\Dim_Job_Roles.csv
 1. Shape: 77 rows | 5 columns
2. Column names are clean.
3. No duplicated rows.
4. No missing values.

🔹 5. Numeric Summary (Min - Mean - Max):
         min  mean   max
role_id 1.00 39.00 77.00

 6. Categorical Summary:
   - job_title: 77 unique values
   - department: 7 unique values
     Values are: ['IT', 'Sales', 'HR', 'Finance', 'Marketing', 'Operations', 'Legal']
   - job_level: 5 unique values
     Values are: ['Junior', 'Mid', 'Senior', 'Lead', 'Director']
   - base_salary_range: 5 unique values
     Values are: ['8000-15000', '15000-25000', '25000-60000', '60000-100000', '100000-250000']




In [44]:
df_roles

,role_id,job_title,department,job_level,base_salary_range
0,1,Junior IT Support,IT,Junior,8000-15000
1,2,Jr. Data Analyst,IT,Junior,8000-15000
2,3,Systems Analyst,IT,Mid,15000-25000
3,4,Software Engineer,IT,Mid,15000-25000
4,5,Senior Data Engineer,IT,Senior,25000-60000
...,...,...,...,...,...
72,73,Sr. Legal Consultant,Legal,Senior,25000-60000
73,74,Legal Team Lead,Legal,Lead,60000-100000
74,75,Compliance Manager,Legal,Lead,60000-100000
75,76,Legal Director,Legal,Director,100000-250000


In [45]:
# Block 5: Audit Fact_Attendance
df_att = audit_table("D:\\HR Project\\HR data\\Raw\\Fact_Attendance.csv")

 Auditing Table: D:\HR Project\HR data\Raw\Fact_Attendance.csv
 1. Shape: 725,408 rows | 7 columns
2. Column names are clean.
3. Fully duplicated rows: 5,757 rows 
4. Missing Values (Nulls):
   - business_travel_frequency: 80,152 nulls (11.05%)

🔹 5. Numeric Summary (Min - Mean - Max):
                       min      mean       max
record_id             1.00 359876.30 719651.00
employee_id           1.00   7523.53  15000.00
total_working_hours 105.00    178.88    899.00
overtime_hours        0.00     22.51    599.00
sick_leaves_taken    -5.00      1.06      8.00

 6. Categorical Summary:
   - month_year: 84 unique values
   - business_travel_frequency: 6 unique values
     Values are: ['Medium', 'High', 'Low', 'low', 'HIGH', 'Med']




In [46]:
df_att

,record_id,employee_id,month_year,total_working_hours,overtime_hours,sick_leaves_taken,business_travel_frequency
0,1,1,2022-10,159,28,1,Medium
1,2,1,2022-11,168,35,4,High
2,3,1,2022-12,199,21,1,High
3,4,1,2023-01,189,13,2,NaN
4,5,1,2023-02,182,15,0,High
...,...,...,...,...,...,...,...
725403,293272,6136,2024-04,191,11,0,High
725404,669153,13956,2021-11,208,25,0,Medium
725405,251015,5256,2025-02,198,18,3,High
725406,665829,13890,2023-02,183,27,1,High


In [47]:
# Block 6: Audit Fact_Compensation
df_comp = audit_table("D:\\HR Project\\HR data\\Raw\\Fact_Compensation.csv")

 Auditing Table: D:\HR Project\HR data\Raw\Fact_Compensation.csv
 1. Shape: 67,769 rows | 6 columns
2. Column names are clean.
3. Fully duplicated rows: 203 rows 
4. No missing values.

🔹 5. Numeric Summary (Min - Mean - Max):
                          min     mean        max
compensation_id          1.00 33780.42   67566.00
employee_id              1.00  7522.04   15000.00
monthly_salary_egp    7982.81 41592.39 1336467.36
annual_bonus_egp   -101390.81 13209.12  328321.18
stock_options            0.00   102.03    1000.00

 6. Categorical Summary:
   - effective_date: 1176 unique values




In [48]:
df_comp

,compensation_id,employee_id,effective_date,monthly_salary_egp,annual_bonus_egp,stock_options
0,1,1,2022-10-28,9794.75,5368.97,0
1,2,1,2023-10-08,9890.30,1336.69,0
2,3,1,2024-04-19,11847.32,5345.91,0
3,4,1,2025-07-07,12554.45,4275.90,0
4,5,2,2020-04-09,83488.62,0.00,179
...,...,...,...,...,...,...
67764,63565,14114,2020-01-08,201968.35,109431.68,356
67765,42301,9410,2022-06-06,48128.81,33757.10,6
67766,43783,9744,2023-10-04,43032.84,24911.13,231
67767,58342,12953,2020-10-26,47303.02,34837.36,310


In [49]:
# Block 7: Audit Fact_Engagement
df_eng = audit_table("D:\\HR Project\\HR data\\Raw\\Fact_Engagement.csv")

 Auditing Table: D:\HR Project\HR data\Raw\Fact_Engagement.csv
 1. Shape: 67,566 rows | 6 columns
2. Column names are clean.
3. No duplicated rows.
4. Missing Values (Nulls):
   - work_life_balance_score: 6,077 nulls (8.99%)
   - manager_satisfaction_score: 8,784 nulls (13.00%)
   - environment_satisfaction: 4,054 nulls (6.00%)

🔹 5. Numeric Summary (Min - Mean - Max):
                            min     mean      max
survey_id                  1.00 33783.50 67566.00
employee_id                1.00  7522.73 15000.00
work_life_balance_score    0.00     2.91     5.00
manager_satisfaction_score 1.00     2.84     5.00
environment_satisfaction   1.00     2.90     5.00

 6. Categorical Summary:
   - survey_date: 567 unique values




In [50]:
df_eng

,survey_id,employee_id,survey_date,work_life_balance_score,manager_satisfaction_score,environment_satisfaction
0,1,1,2022-06-16,3.00,4.00,3.00
1,2,1,2023-05-16,1.00,NaN,3.00
2,3,1,2024-07-04,4.00,NaN,4.00
3,4,1,2025-06-03,3.00,2.00,3.00
4,5,2,2020-06-28,2.00,3.00,3.00
...,...,...,...,...,...,...
67561,67562,14999,2024-07-08,3.00,3.00,3.00
67562,67563,14999,2025-06-01,2.00,NaN,4.00
67563,67564,15000,2023-05-02,3.00,3.00,3.00
67564,67565,15000,2024-06-21,1.00,NaN,3.00


In [51]:
# Block 8: Audit Fact_Performance
df_perf = audit_table("D:\\HR Project\\HR data\\Raw\\Fact_Performance.csv")

 Auditing Table: D:\HR Project\HR data\Raw\Fact_Performance.csv
 1. Shape: 67,904 rows | 5 columns
2. Column names are clean.
3. Fully duplicated rows: 338 rows 
4. No missing values.

🔹 5. Numeric Summary (Min - Mean - Max):
                    min     mean      max
review_id          1.00 33792.14 67566.00
employee_id        1.00  7524.64 15000.00
performance_score  0.00     2.85     6.00
promoted_this_year 0.00     0.10     1.00

 6. Categorical Summary:
   - review_date: 637 unique values




In [52]:
df_perf

,review_id,employee_id,review_date,performance_score,promoted_this_year
0,1,1,2022-11-19,2,0
1,2,1,2023-12-15,2,0
2,3,1,2024-12-14,2,1
3,4,1,2025-10-21,3,0
4,5,2,2020-10-26,3,0
...,...,...,...,...,...
67899,65348,14509,2024-12-22,4,0
67900,65244,14490,2024-01-10,5,1
67901,46057,10230,2024-12-20,3,1
67902,41155,9157,2020-10-22,2,0
